# Entity ***Factory***
+ Layer bronze
+ Priority 1
---
### Imports

In [0]:
%run ../00_functions/medallion_functions

In [0]:
import requests
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
dbutils.widgets.text("layer", "bronze")
dbutils.widgets.text("entity_name", "factory")
dbutils.widgets.text("load_pattern", "1")
dbutils.widgets.text("SUBGRAPH_URL", "https://gateway.thegraph.com/api/a5bbc98aac75dac555f9aba8747c7e2e/subgraphs/id/5zvR82QoaXYFyDEKLZ9t6v9adgnptxYpKpSbxtgVENFV")

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")
SUBGRAPH_URL = dbutils.widgets.get("SUBGRAPH_URL")

### Variables

In [0]:
query_variables = {
    "MAINNET_FACTORY_ID": "0x1F98431c8aD98523631AE4a59f267346ea31F984"
}


In [0]:
factory_query = """
query FactoryQuery($MAINNET_FACTORY_ID: ID!){
    factory(id: $MAINNET_FACTORY_ID ) {
    id
    poolCount
    txCount
    totalFeesUSD
    totalFeesETH
    totalVolumeUSD
    totalVolumeETH
    totalValueLockedUSD
    totalValueLockedETH
  }
}
"""

### Execution

In [0]:
response = requests.post(SUBGRAPH_URL, json={"query": factory_query, "variables": query_variables})

In [0]:
factory_df = spark.createDataFrame([response.json()["data"][entity_name]])
factory_df = factory_df.withColumn("load_timestamp", current_timestamp())

### Save & exit

In [0]:
load_result = load_entity(factory_df,
                        entity_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)